# Qwen3.5-4B RotQuant LoRA trial matrix

This notebook runs the complete evidence ladder needed to decide whether LoRA should be retained on the 4-bit RotQuant model. It establishes matched CUDA controls, increases distillation data before capacity, conditionally escalates from rank 4 to rank 8, validates the winning 4-bit recipe across seeds, and exports one compact report. It also runs one seed-0 3-bit LoRA-QAT arm to measure the more aggressive quality/size tradeoff.

It is intentionally **not** a standard SFT notebook. RotQuant keeps packed codes fixed and trains low-rank recovery adapters against source-model logits so the experiment measures quantization recovery rather than unrelated task fine-tuning.

## Goal

Select the smallest deployed candidate satisfying:

- Mean perplexity degradation across seeds no greater than **5%**.
- Worst-seed degradation no greater than **10%**.
- LoRA is retained only if its disjoint held-out gate accepts it and it improves deployed WikiText-2 perplexity by a meaningful amount.
- All size calculations include retained butterfly parameters and adapters.

### Key assumptions

- Training, validation, and final selection use disjoint C4 sequences; WikiText-2 test data is evaluation-only.
- The language decoder is quantized. The vision tower, tied output head, and small recurrent-state gate projections remain in source precision.
- Cached FP16 fallback is enabled to accelerate quality training on the RTX PRO 6000-class GPU. Its VRAM is not the packed deployment footprint.
- Result JSON and reports are persisted, but the repository still lacks reloadable packed-checkpoint export.
- This notebook can take several hours. It resumes completed trial directories unless `FORCE_RERUN=True`.

## Trial matrix

| Stage | Trial | Purpose |
|---|---|---|
| Controls | Source CUDA | Matched unquantized reference |
| Controls | Plain 4-bit FWHT | Isolate the value of training |
| Controls | Block-only | Measure butterfly/scale recovery without LoRA |
| LoRA A | Rank 4, 8/2/4 batches | Add data before capacity |
| 3-bit probe | Rank 4, 8/2/4 batches, seed 0 | Test aggressive compression with LoRA-QAT |
| LoRA B | Rank 4, 16/4/4 batches | Conditional overfitting remedy |
| LoRA C | Rank 8, 16/4/4 batches | Conditional capacity remedy |
| Release | Winning recipe, seeds 1 and 2 | Estimate stability |

The 3-bit probe always runs once and is reported separately; it cannot displace the 4-bit release winner without its own multi-seed confirmation. LoRA B and C run only when the preceding 4-bit candidate is rejected or fails to beat block-only PPL. Set `FORCE_ALL_LORA_TRIALS=True` to run every 4-bit arm regardless.

## Setup

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/CodeHalwell/rotquant.git"
REPO_REF = "main"
REPO_DIR = Path("/content/rotquant-lora-matrix")
MODEL_ID = "unsloth/Qwen3.5-4B"
CONFIG_RELATIVE_PATH = Path("configs/qwen35_4b_lora_qat_cuda.yaml")

# Use Drive so JSON results survive a runtime disconnect. Disable only when
# running outside Colab or when you will download the final archive immediately.
USE_GOOGLE_DRIVE = True
DRIVE_RESULT_ROOT = Path("/content/drive/MyDrive/rotquant/qwen35_lora_matrix")
LOCAL_RESULT_ROOT = Path("/content/qwen35_lora_matrix")

EVAL_SEQ_LEN = 256
EVAL_MAX_SAMPLES = 128
MEAN_PPL_GATE = 0.05
WORST_PPL_GATE = 0.10
MIN_LORA_PPL_IMPROVEMENT = 0.05  # Absolute PPL needed to justify serving LoRA.

CONFIRM_EXPENSIVE_RUN = False  # Read the matrix, then set True.
FORCE_RERUN = False
FORCE_ALL_LORA_TRIALS = False
RUN_3BIT_LORA_PROBE = True
RUN_SEED_VALIDATION = True
EXPORT_WINNER = True  # Reruns only the selected seed-0 recipe and saves it.
EXPORT_PROCESSOR = True  # Preserve Qwen vision/text processor metadata.
DOWNLOAD_RESULTS = True
TRACKIO_SPACE_ID = None  # Optional: "your-hf-username/trackio".

print({
    "repo_ref": REPO_REF,
    "eval_samples": EVAL_MAX_SAMPLES,
    "confirm_expensive_run": CONFIRM_EXPENSIVE_RUN,
})

### 1. Verify the GPU

In [ ]:
import os
import subprocess
import sys
import torch

assert torch.cuda.is_available(), "Select a CUDA GPU runtime before continuing."
gpu = torch.cuda.get_device_properties(0)
vram_gib = gpu.total_memory / 2**30
print(f"GPU: {gpu.name}")
print(f"VRAM: {vram_gib:.1f} GiB | torch={torch.__version__} | CUDA={torch.version.cuda}")
if vram_gib < 40:
    print("WARNING: cached fallback may OOM below 40 GiB.")
subprocess.run(["nvidia-smi"], check=True)

### 2. Persist results and fetch the repository

In [ ]:
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RESULT_BASE = DRIVE_RESULT_ROOT
else:
    RESULT_BASE = LOCAL_RESULT_ROOT
RESULT_BASE.mkdir(parents=True, exist_ok=True)
print(f"Persistent result root: {RESULT_BASE}")

In [ ]:
if not REPO_DIR.exists():
    subprocess.run([
        "git", "clone", "--branch", REPO_REF, "--single-branch",
        REPO_URL, str(REPO_DIR),
    ], check=True)
else:
    print(f"Reusing {REPO_DIR}; delete it to clone a newer revision.")

required_paths = [
    REPO_DIR / "rotquant/block_train.py",
    REPO_DIR / CONFIG_RELATIVE_PATH,
    REPO_DIR / "scripts/run_experiment.py",
]
missing = [str(path) for path in required_paths if not path.exists()]
assert not missing, "Missing required repository files: " + ", ".join(missing)
commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True
).strip()
RESULT_ROOT = RESULT_BASE / commit[:12]
RESULT_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Using commit {commit}; results: {RESULT_ROOT}")

### 3. Install the runtime without replacing CUDA PyTorch

In [ ]:
runtime_packages = [
    "transformers>=5.9,<6",
    "datasets>=4.8",
    "accelerate",
    "safetensors",
    "sentencepiece",
    "scipy",
    "pyyaml",
    "pandas",
    "matplotlib",
    "huggingface_hub",
    "trackio",
]
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-U", *runtime_packages],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR), "--no-deps"],
    check=True,
)

os.environ["TORCH_ALLOW_TF32_CUBLAS_OVERRIDE"] = "1"
torch.backends.cuda.matmul.allow_tf32 = True
torch.set_float32_matmul_precision("high")

import transformers
from transformers import AutoModelForMultimodalLM
print(f"transformers={transformers.__version__}; multimodal loader available")

### 4. Validate the base configuration

In [ ]:
import yaml

config_path = REPO_DIR / CONFIG_RELATIVE_PATH
with config_path.open() as handle:
    experiment_config = yaml.safe_load(handle)
assert experiment_config["model"] == MODEL_ID
assert experiment_config["device"] == "cuda"
assert experiment_config["quant"]["bits"] == 4
assert experiment_config["patch"]["fallback"] is True
assert experiment_config["patch"]["train_rotation"]["distill_lora_rank"] == 4
print(yaml.safe_dump(experiment_config, sort_keys=False))

## Steps

### 5. Define resumable trial and diagnostic helpers

In [ ]:
import hashlib
import json
import shlex
from typing import Iterable

trial_records = {}

def _latest_result(output_dir: Path):
    candidates = sorted(output_dir.glob("*.json"), key=lambda path: path.stat().st_mtime)
    if not candidates:
        return None
    with candidates[-1].open() as handle:
        return json.load(handle)

def run_trial(trial_name: str, overrides: Iterable[str], seed: int = 0):
    overrides = list(overrides)
    signature = hashlib.sha256(json.dumps({
        "seed": seed, "overrides": overrides,
    }, sort_keys=True).encode()).hexdigest()[:10]
    output_dir = RESULT_ROOT / f"{trial_name}_{signature}"
    output_dir.mkdir(parents=True, exist_ok=True)
    if not FORCE_RERUN:
        cached = _latest_result(output_dir)
        if cached is not None:
            print(f"Reusing completed trial: {trial_name}")
            trial_records[trial_name] = cached
            return cached

    command = [
        sys.executable,
        str(REPO_DIR / "scripts/run_experiment.py"),
        str(config_path),
        "--output-dir", str(output_dir),
        "--seed", str(seed),
    ]
    for override in overrides:
        command.extend(["--set", override])
    print("Running:", shlex.join(command), flush=True)
    subprocess.run(command, cwd=REPO_DIR, env=os.environ.copy(), check=True)
    payload = _latest_result(output_dir)
    assert payload is not None, f"No result JSON was written to {output_dir}"
    trial_records[trial_name] = payload
    return payload

def distillation_stats(payload):
    return payload.get("metrics", {}).get("distillation", {})

def lora_is_useful(payload, block_ppl: float):
    metrics = payload["metrics"]
    stats = distillation_stats(payload)
    return bool(
        stats.get("lora_retained")
        and stats.get("best_step", 0) > 0
        and metrics["ppl_wikitext2"] <= block_ppl - MIN_LORA_PPL_IMPROVEMENT
    )

matched_eval = [
    f"eval.ppl.seq_len={EVAL_SEQ_LEN}",
    f"eval.ppl.max_samples={EVAL_MAX_SAMPLES}",
]

In [ ]:
plain_4bit_overrides = [
    "patch.rotation=fwht",
    "patch.train_rotation=null",
    *matched_eval,
]
block_only_overrides = [
    "patch.train_rotation.distill_steps=0",
    *matched_eval,
]
rank4_medium_overrides = [
    "patch.train_rotation.distill_steps=24",
    "patch.train_rotation.distill_train_batches=8",
    "patch.train_rotation.distill_validation_batches=2",
    "patch.train_rotation.distill_selection_batches=4",
    "patch.train_rotation.distill_lora_rank=4",
    "patch.train_rotation.distill_lora_alpha=8.0",
    "patch.train_rotation.distill_lora_lr=0.0005",
    "patch.train_rotation.distill_early_stopping_patience=6",
    "patch.train_rotation.distill_selection_min_improvement=0.001",
    *matched_eval,
]
threebit_rank4_medium_overrides = [
    "quant.bits=3",
    *rank4_medium_overrides,
]
rank4_large_overrides = [
    "patch.train_rotation.distill_steps=32",
    "patch.train_rotation.distill_train_batches=16",
    "patch.train_rotation.distill_validation_batches=4",
    "patch.train_rotation.distill_selection_batches=4",
    "patch.train_rotation.distill_lora_rank=4",
    "patch.train_rotation.distill_lora_alpha=8.0",
    "patch.train_rotation.distill_lora_lr=0.0003",
    "patch.train_rotation.distill_early_stopping_patience=8",
    "patch.train_rotation.distill_selection_min_improvement=0.001",
    *matched_eval,
]
rank8_large_overrides = [
    "patch.train_rotation.distill_steps=32",
    "patch.train_rotation.distill_train_batches=16",
    "patch.train_rotation.distill_validation_batches=4",
    "patch.train_rotation.distill_selection_batches=4",
    "patch.train_rotation.distill_lora_rank=8",
    "patch.train_rotation.distill_lora_alpha=16.0",
    "patch.train_rotation.distill_lora_lr=0.0003",
    "patch.train_rotation.distill_early_stopping_patience=8",
    "patch.train_rotation.distill_selection_min_improvement=0.001",
    *matched_eval,
]
trial_profiles = {
    "block_only": block_only_overrides,
    "rank4_medium": rank4_medium_overrides,
    "rank4_large": rank4_large_overrides,
    "rank8_large": rank8_large_overrides,
}

### 6. Confirm the expensive matrix

Set `CONFIRM_EXPENSIVE_RUN=True` in the parameter cell after checking the GPU and persistence path.

In [ ]:
assert CONFIRM_EXPENSIVE_RUN, (
    "Set CONFIRM_EXPENSIVE_RUN=True before launching the multi-hour trial matrix."
)

### 7. Run matched controls

In [ ]:
source_result = run_trial(
    "source_cuda_s0",
    ["patch.enabled=false", *matched_eval],
    seed=0,
)
source_ppl = source_result["metrics"]["ppl_wikitext2"]
print(f"Source CUDA PPL: {source_ppl:.4f}")

In [ ]:
plain_result = run_trial("plain_4bit_fwht_s0", plain_4bit_overrides, seed=0)
block_result = run_trial("block_only_s0", block_only_overrides, seed=0)
plain_ppl = plain_result["metrics"]["ppl_wikitext2"]
block_ppl = block_result["metrics"]["ppl_wikitext2"]
print({
    "plain_4bit_ppl": plain_ppl,
    "block_only_ppl": block_ppl,
    "block_training_gain": plain_ppl - block_ppl,
})

### 8. Run rank-4 with more data first

In [ ]:
rank4_medium_result = run_trial(
    "rank4_medium_s0", rank4_medium_overrides, seed=0
)
rank4_medium_stats = distillation_stats(rank4_medium_result)
rank4_medium_useful = lora_is_useful(rank4_medium_result, block_ppl)
print(json.dumps(rank4_medium_stats, indent=2))
print({
    "deployed_ppl": rank4_medium_result["metrics"]["ppl_wikitext2"],
    "useful_lora": rank4_medium_useful,
})

### 8a. Run one 3-bit LoRA-QAT probe

This uses the same rank-4 medium-data recipe as LoRA A with `quant.bits=3`. It is a one-seed frontier probe, not a release candidate; promote it to seeds 1 and 2 only if its seed-0 result is compelling.

In [ ]:
threebit_result = None
if RUN_3BIT_LORA_PROBE:
    threebit_result = run_trial(
        "threebit_rank4_medium_s0",
        threebit_rank4_medium_overrides,
        seed=0,
    )
    threebit_stats = distillation_stats(threebit_result)
    threebit_ppl = threebit_result["metrics"]["ppl_wikitext2"]
    print(json.dumps(threebit_stats, indent=2))
    print({
        "deployed_ppl": threebit_ppl,
        "relative_to_source": threebit_ppl / source_ppl - 1.0,
        "lora_retained": bool(threebit_stats.get("lora_retained", False)),
    })
else:
    print("3-bit LoRA-QAT probe disabled.")

### 9. Conditionally increase data, then rank

The larger rank-4 trial comes before rank 8 because held-out rejection after strong training improvement usually indicates data variance or overfitting, not insufficient adapter capacity.

In [ ]:
rank4_large_result = None
if FORCE_ALL_LORA_TRIALS or not rank4_medium_useful:
    rank4_large_result = run_trial(
        "rank4_large_s0", rank4_large_overrides, seed=0
    )
    rank4_large_stats = distillation_stats(rank4_large_result)
    rank4_large_useful = lora_is_useful(rank4_large_result, block_ppl)
    print(json.dumps(rank4_large_stats, indent=2))
    print({
        "deployed_ppl": rank4_large_result["metrics"]["ppl_wikitext2"],
        "useful_lora": rank4_large_useful,
    })
else:
    rank4_large_useful = False
    print("Rank-4 medium passed; larger-data rank-4 trial not needed.")

In [ ]:
rank8_large_result = None
rank4_any_useful = rank4_medium_useful or rank4_large_useful
if FORCE_ALL_LORA_TRIALS or not rank4_any_useful:
    rank8_large_result = run_trial(
        "rank8_large_s0", rank8_large_overrides, seed=0
    )
    rank8_large_stats = distillation_stats(rank8_large_result)
    rank8_large_useful = lora_is_useful(rank8_large_result, block_ppl)
    print(json.dumps(rank8_large_stats, indent=2))
    print({
        "deployed_ppl": rank8_large_result["metrics"]["ppl_wikitext2"],
        "useful_lora": rank8_large_useful,
    })
else:
    rank8_large_useful = False
    print("A rank-4 recipe passed; rank-8 trial not needed.")

### 10. Select the seed-0 winner

Rejected LoRA candidates are not eligible as LoRA recipes; their deployed output is simply another copy of the block model. Accepted LoRA must also beat block-only by `MIN_LORA_PPL_IMPROVEMENT`.

In [ ]:
eligible = [{
    "profile": "block_only",
    "payload": block_result,
    "overrides": block_only_overrides,
}]
for profile, payload, useful in [
    ("rank4_medium", rank4_medium_result, rank4_medium_useful),
    ("rank4_large", rank4_large_result, rank4_large_useful),
    ("rank8_large", rank8_large_result, rank8_large_useful),
]:
    if payload is not None and useful:
        eligible.append({
            "profile": profile,
            "payload": payload,
            "overrides": trial_profiles[profile],
        })
winner = min(eligible, key=lambda item: item["payload"]["metrics"]["ppl_wikitext2"] )
winner_profile = winner["profile"]
winner_result = winner["payload"]
winner_overrides = winner["overrides"]
winner_ppl = winner_result["metrics"]["ppl_wikitext2"]
print({
    "winner_profile": winner_profile,
    "winner_seed0_ppl": winner_ppl,
    "relative_to_source": winner_ppl / source_ppl - 1.0,
    "lora_retained": distillation_stats(winner_result).get("lora_retained", False),
})

### 11. Validate the winning recipe across seeds

In [ ]:
seed_results = {0: winner_result}
if RUN_SEED_VALIDATION:
    for seed in (1, 2):
        seed_results[seed] = run_trial(
            f"winner_{winner_profile}_s{seed}",
            winner_overrides,
            seed=seed,
        )
else:
    print("Seed validation disabled; do not treat a single-seed pass as final.")

## Checks

### 12. Build the complete quality/size report

In [ ]:
import numpy as np
import pandas as pd
from huggingface_hub import hf_hub_download

index_path = hf_hub_download(MODEL_ID, "model.safetensors.index.json")
with open(index_path) as handle:
    source_weight_bytes = json.load(handle)["metadata"]["total_size"]

def estimated_complete_bytes(payload):
    metrics = payload["metrics"]
    if "fp16_weight_bytes" not in metrics:
        return source_weight_bytes
    deployed_quantized = metrics.get(
        "packed_plus_auxiliary_bytes", metrics["packed_weight_bytes"]
    )
    return source_weight_bytes - metrics["fp16_weight_bytes"] + deployed_quantized

def report_row(name, payload, seed=0):
    metrics = payload["metrics"]
    distill = distillation_stats(payload)
    size_bytes = estimated_complete_bytes(payload)
    return {
        "trial": name,
        "seed": seed,
        "ppl": metrics["ppl_wikitext2"],
        "relative_ppl": metrics["ppl_wikitext2"] / source_ppl - 1.0,
        "estimated_GB": size_bytes / 1e9,
        "size_reduction": 1.0 - size_bytes / source_weight_bytes,
        "lora_retained": bool(distill.get("lora_retained", False)),
        "adapter_MB": metrics.get("adapter_parameter_bytes", 0) / 1e6,
        "selection_device": metrics.get("rotation_train", {}).get("selection_device"),
    }

rows = [
    report_row("source_cuda", source_result),
    report_row("plain_4bit_fwht", plain_result),
    report_row("block_only", block_result),
    report_row("rank4_medium", rank4_medium_result),
]
if threebit_result is not None:
    rows.append(report_row("threebit_rank4_medium", threebit_result))
if rank4_large_result is not None:
    rows.append(report_row("rank4_large", rank4_large_result))
if rank8_large_result is not None:
    rows.append(report_row("rank8_large", rank8_large_result))
for seed, payload in seed_results.items():
    if seed:
        rows.append(report_row(f"winner_{winner_profile}", payload, seed=seed))
report = pd.DataFrame(rows)
display(report.style.format({
    "ppl": "{:.4f}",
    "relative_ppl": "{:+.2%}",
    "estimated_GB": "{:.3f}",
    "size_reduction": "{:.2%}",
    "adapter_MB": "{:.2f}",
}))
report_path = RESULT_ROOT / "trial_matrix.csv"
report.to_csv(report_path, index=False)
print(f"Wrote {report_path}")

In [ ]:
import matplotlib.pyplot as plt

plot_rows = report.drop_duplicates(subset=["trial", "seed"])
fig, ax = plt.subplots(figsize=(9, 5))
for _, row in plot_rows.iterrows():
    ax.scatter(row["estimated_GB"], row["ppl"], s=70)
    ax.annotate(
        f"{row['trial']} s{row['seed']}",
        (row["estimated_GB"], row["ppl"]),
        xytext=(5, 5), textcoords="offset points", fontsize=8,
    )
ax.axhline(source_ppl * (1 + MEAN_PPL_GATE), color="tab:orange", linestyle="--", label="5% PPL gate")
ax.axhline(source_ppl * (1 + WORST_PPL_GATE), color="tab:red", linestyle=":", label="10% PPL gate")
ax.set_xlabel("Estimated complete-model weight size (GB)")
ax.set_ylabel("WikiText-2 perplexity (lower is better)")
ax.set_title("Qwen3.5-4B RotQuant quality-size frontier")
ax.grid(alpha=0.25)
ax.legend()
plt.tight_layout()
plot_path = RESULT_ROOT / "quality_size_frontier.png"
fig.savefig(plot_path, dpi=160)
plt.show()
print(f"Wrote {plot_path}")

### 13. Apply the release rule

In [ ]:
winner_ppls = np.array([
    payload["metrics"]["ppl_wikitext2"] for payload in seed_results.values()
])
winner_relative = winner_ppls / source_ppl - 1.0
mean_relative = float(winner_ppls.mean() / source_ppl - 1.0)
worst_relative = float(winner_relative.max())
lora_required = winner_profile != "block_only"
lora_retention_rate = float(np.mean([
    bool(distillation_stats(payload).get("lora_retained", False))
    for payload in seed_results.values()
])) if lora_required else 1.0
release_pass = bool(
    len(seed_results) >= 3
    and mean_relative <= MEAN_PPL_GATE
    and worst_relative <= WORST_PPL_GATE
    and lora_retention_rate == 1.0
)
threebit_summary = None
if threebit_result is not None:
    threebit_ppl = threebit_result["metrics"]["ppl_wikitext2"]
    threebit_summary = {
        "seed": 0,
        "ppl": threebit_ppl,
        "relative_ppl": threebit_ppl / source_ppl - 1.0,
        "estimated_GB": estimated_complete_bytes(threebit_result) / 1e9,
        "lora_retained": bool(
            distillation_stats(threebit_result).get("lora_retained", False)
        ),
        "within_10pct_probe_gate": threebit_ppl <= source_ppl * (1 + WORST_PPL_GATE),
    }
release_summary = {
    "winner_profile": winner_profile,
    "seeds": sorted(seed_results),
    "mean_ppl": float(winner_ppls.mean()),
    "std_ppl": float(winner_ppls.std(ddof=1)) if len(winner_ppls) > 1 else 0.0,
    "mean_relative_ppl": mean_relative,
    "worst_relative_ppl": worst_relative,
    "lora_retention_rate": lora_retention_rate,
    "threebit_probe": threebit_summary,
    "release_pass": release_pass,
}
print(json.dumps(release_summary, indent=2))
with (RESULT_ROOT / "release_summary.json").open("w") as handle:
    json.dump(release_summary, handle, indent=2)

### 14. Export the winning packed model

The trial subprocesses exit after writing metrics, so the selected seed-0 recipe must be reconstructed once to serialize it. Evaluation is disabled for this export-only rerun. The artifact is written outside `RESULT_ROOT` so the small report archive does not duplicate a roughly 4 GB model.

In [ ]:
PACKED_EXPORT_DIR = RESULT_BASE / (
    f"{commit[:12]}_{winner_profile}_s0_packed"
)
packed_manifest_path = PACKED_EXPORT_DIR / "rotquant_config.json"
if EXPORT_WINNER:
    assert release_pass, "Refusing to export a recipe that failed the release gate."
    if packed_manifest_path.exists():
        print(f"Reusing packed checkpoint: {PACKED_EXPORT_DIR}")
    else:
        export_run_dir = RESULT_ROOT / "export_run"
        export_run_dir.mkdir(parents=True, exist_ok=True)
        export_command = [
            sys.executable,
            str(REPO_DIR / "scripts/run_experiment.py"),
            str(config_path),
            "--output-dir", str(export_run_dir),
            "--seed", "0",
            "--export-dir", str(PACKED_EXPORT_DIR),
        ]
        if EXPORT_PROCESSOR:
            export_command.append("--export-processor")
        for override in [
            *winner_overrides,
            "eval.perplexity=false",
            "eval.zeroshot=false",
        ]:
            export_command.extend(["--set", override])
        print("Running:", shlex.join(export_command), flush=True)
        subprocess.run(
            export_command, cwd=REPO_DIR, env=os.environ.copy(), check=True
        )
    assert packed_manifest_path.exists(), "Packed export did not complete."
    with packed_manifest_path.open() as handle:
        packed_manifest = json.load(handle)
    print({
        "packed_checkpoint": str(PACKED_EXPORT_DIR),
        "format_version": packed_manifest["format_version"],
        "quantized_modules": len(packed_manifest["quantized_modules"]),
        "fallback_cache_serialized": False,
    })
else:
    print("Packed winner export disabled.")

### 15. Optionally publish the compact report to Trackio

In [ ]:
if TRACKIO_SPACE_ID:
    import trackio
    trackio.init(
        project="rotquant-qwen35-lora-matrix",
        name=f"winner-{winner_profile}-{commit[:8]}",
        space_id=TRACKIO_SPACE_ID,
        config={
            "model": MODEL_ID,
            "bits": 4,
            "eval_samples": EVAL_MAX_SAMPLES,
            "winner_profile": winner_profile,
        },
    )
    trackio.log(release_summary)
    trackio.finish()
    print(f"Published summary to {TRACKIO_SPACE_ID}")
else:
    print("Trackio publishing skipped.")

### 16. Download the compact experiment record

In [ ]:
import shutil

archive_base = Path("/content/qwen35_lora_trial_matrix")
archive_path = Path(shutil.make_archive(str(archive_base), "zip", RESULT_ROOT))
print(f"Created {archive_path} ({archive_path.stat().st_size / 1e6:.2f} MB)")
if DOWNLOAD_RESULTS:
    from google.colab import files
    files.download(str(archive_path))

## Next steps

- If `release_pass=true`, retain the Drive-backed packed checkpoint and optionally upload that directory to a private Hugging Face repository.
- Load it with `rotquant.checkpoint.load_packed_model`; use `AutoTokenizer.from_pretrained` or `AutoProcessor.from_pretrained` on the same directory.
- If LoRA retention varies by seed, prefer block-only; an intermittently retained adapter is not a stable deployment recipe.
- If all LoRA arms are rejected while block-only passes, that is a successful negative result: extra adapter storage and serving complexity are unnecessary.
- If the winner fails the mean gate, increase representative C4 calibration data before increasing steps or rank again. Do not train on WikiText-2 test data.